In [1]:
# Imports (todos nesta célula)
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "repeat_purchase_90d"
GROUP = "customer_id"
DATE = "purchase_date"
OUT_OF_MODEL = [GROUP, DATE]   # ficam na tabela, mas nunca entram no modelo
METRIC = "f1"                  # métrica que decide as comparações (accuracy ao lado)

In [2]:
# ===== [2.0] BASE COMUM - início =====
# Passo 1 - dados de partida, ordenados por data como na W5 (secção 3.4).
# As linhas sem data não têm lugar na cronologia, por isso ficam de fora.
raw = pd.read_csv("../data/train.csv", index_col="ID")

known_time = raw[DATE].notna()
# kind="stable" mantém pela ordem do ficheiro as compras do mesmo dia (W5)
time_order = raw.loc[known_time, DATE].sort_values(kind="stable").index
frame = raw.loc[time_order]

print("linhas excluídas por não terem data:", int((~known_time).sum()))
print(f"linhas na cronologia: {len(frame):,}, de {frame[DATE].iloc[0]} a {frame[DATE].iloc[-1]}")

linhas excluídas por não terem data: 65
linhas na cronologia: 6,469, de 2015-07-11 a 2025-02-15


In [3]:
# Passo 2 - que pergunta faz o test.csv? Só a estrutura (datas e clientes); nenhum valor
# do teste é usado para ajustar nada. As datas comparam-se como texto, como na EDA.
test_csv = pd.read_csv("../data/test.csv", index_col="ID")
test_rows_seen = test_csv[GROUP].isin(raw[GROUP])

pd.Series({
    "datas do treino": f"{frame[DATE].iloc[0]} a {frame[DATE].iloc[-1]}",
    "datas do teste": f"{test_csv[DATE].min()} a {test_csv[DATE].max()}",
    "clientes do teste já vistos no treino":
        f"{pd.Series(test_csv[GROUP].unique()).isin(raw[GROUP]).sum()} de {test_csv[GROUP].nunique()}",
    "linhas do teste com cliente já visto":
        f"{test_rows_seen.sum()} de {len(test_csv)} ({100 * test_rows_seen.mean():.1f}%)",
}).to_frame("o que o teste pergunta")

,o que o teste pergunta
datas do treino,2015-07-11 a 2025-02-15
datas do teste,2025-05-24 a 2026-10-06
clientes do teste já vistos no treino,184 de 206
linhas do teste com cliente já visto,932 de 986 (94.5%)


In [4]:
# Passo 3 - divisões por tempo: cada divisão treina no passado e valida no que vem a seguir,
# como o test.csv, que tem compras posteriores às do train.csv
tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(frame))

for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"divisão {fold}: treino até {frame[DATE].iloc[train_idx].max()}"
          f" ({len(train_idx):,} linhas), valida de {frame[DATE].iloc[val_idx].min()}"
          f" a {frame[DATE].iloc[val_idx].max()} ({len(val_idx):,} linhas)")

divisão 1: treino até 2017-08-12 (1,079 linhas), valida de 2017-08-12 a 2019-01-12 (1,078 linhas)
divisão 2: treino até 2019-01-12 (2,157 linhas), valida de 2019-01-12 a 2020-10-03 (1,078 linhas)
divisão 3: treino até 2020-10-03 (3,235 linhas), valida de 2020-10-03 a 2022-03-05 (1,078 linhas)
divisão 4: treino até 2022-03-05 (4,313 linhas), valida de 2022-03-05 a 2023-09-25 (1,078 linhas)
divisão 5: treino até 2023-09-25 (5,391 linhas), valida de 2023-09-25 a 2025-02-15 (1,078 linhas)


In [5]:
# Passo 4 - funções: preparação da linha de base, avaliação e comparação
def model_columns(frame):
    """Colunas que o modelo vê: numéricas e categóricas (sem alvo, cliente e data)."""
    features = [c for c in frame.columns if c not in OUT_OF_MODEL + [TARGET]]
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(frame[c])]
    categorical = [c for c in features if c not in numeric]
    return numeric, categorical


def as_object(block):
    """O scikit-learn lê np.nan como nulo, mas não pd.NA: tudo passa a np.nan (W3)."""
    return block.astype(object).fillna(np.nan).infer_objects()


def prepare_base(train, test):
    """Linha de base: mediana nas numéricas e moda nas categóricas, ajustadas só no treino."""
    train, test = train.copy(), test.copy()
    numeric, categorical = model_columns(train)
    imputer = SimpleImputer(strategy="median").fit(train[numeric])
    train[numeric], test[numeric] = imputer.transform(train[numeric]), imputer.transform(test[numeric])
    imputer = SimpleImputer(strategy="most_frequent").fit(as_object(train[categorical]))
    train[categorical] = imputer.transform(as_object(train[categorical]))
    test[categorical] = imputer.transform(as_object(test[categorical]))
    return train, test


def evaluate(prepare, frame, splits, scaler=StandardScaler()):
    """F1 e accuracy de uma alternativa `prepare(train, test)` em cada divisão.

    Em cada divisão, tudo é ajustado só no treino: `prepare`, o one-hot e o
    scaler. `scaler=None` não escala. Como o `held_out_f1` da W3.
    """
    rows = []
    for train_idx, val_idx in splits:
        train, test = prepare(frame.iloc[train_idx], frame.iloc[val_idx])
        numeric, categorical = model_columns(train)
        # handle_unknown="ignore": um nível que só aparece na validação fica a zeros
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        encoder.fit(as_object(train[categorical]))
        a = np.hstack([train[numeric], encoder.transform(as_object(train[categorical]))])
        b = np.hstack([test[numeric], encoder.transform(as_object(test[categorical]))])
        if scaler is not None:
            scaler.fit(a)
            a, b = scaler.transform(a), scaler.transform(b)
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(a, train[TARGET])
        predicted = model.predict(b)
        rows.append({"f1": f1_score(test[TARGET], predicted),
                     "accuracy": accuracy_score(test[TARGET], predicted)})
    return pd.DataFrame(rows)


def sem(values):
    """Erro-padrão da média: quanto a média mudaria com outras divisões (W3)."""
    return values.std(ddof=1) / np.sqrt(len(values))


def compare(runs, reference, metric=METRIC):
    """Cada alternativa contra a linha de base, nas mesmas divisões (como na W3).

    Uma diferença menor do que 2 x SEM é ruído.
    """
    table = {}
    for name, run in runs.items():
        difference = run[metric] - reference[metric]
        table[name] = {metric: run[metric].mean(),
                       "accuracy": run["accuracy"].mean(),
                       "vs base": difference.mean(),
                       "SEM": sem(difference),
                       "melhor em": f"{int((difference > 0).sum())} de {len(difference)}"}
    return pd.DataFrame(table).T

In [6]:
# Passo 5 - linha de base: mediana / moda, one-hot, StandardScaler, sem features novas
baseline = evaluate(prepare_base, frame, splits)

print(f"linha de base: F1 {baseline['f1'].mean():.4f} +- {sem(baseline['f1']):.4f}"
      f" | accuracy {baseline['accuracy'].mean():.4f} +- {sem(baseline['accuracy']):.4f}")
baseline.round(4)

linha de base: F1 0.7321 +- 0.0090 | accuracy 0.7056 +- 0.0035


,f1,accuracy
0,0.7543,0.6929
1,0.7258,0.7106
2,0.7350,0.7124
3,0.7015,0.7032
4,0.7439,0.7087


In [7]:
# Passo 6 - o piso: prever sempre 1 (a classe maioritária) nas mesmas linhas de validação
always_one = pd.DataFrame({
    "f1": [f1_score(frame[TARGET].iloc[v], np.ones(len(v), dtype=int)) for _, v in splits],
    "accuracy": [frame[TARGET].iloc[v].mean() for _, v in splits],
})

pd.DataFrame({"linha de base": baseline.mean(), "prever sempre 1": always_one.mean()}).round(4)

,linha de base,prever sempre 1
f1,0.7321,0.7206
accuracy,0.7056,0.5638


In [8]:
# ===== [2.0] BASE COMUM - fim =====
# Como comparar uma alternativa daqui em diante (pacotes 2.1 a 2.6):
#
#   def prepare_a(train, test):      # ajusta tudo só em `train`; devolve (train, test) sem nulos
#       ...
#   compare({"alternativa A": evaluate(prepare_a, frame, splits)}, baseline)
#
# Para comparar escalas (2.4): evaluate(prepare_base, frame, splits, scaler=None) ou outro scaler.

In [9]:
# ===== [2.1] VALORES IMPOSSÍVEIS - início =====
# Passo 7 - regras de domínio da EDA (passos 9 a 11): True onde o valor é impossível
invalid_errors = {
    "days_since_previous_purchase": lambda values: values < 0,          # dias negativos (-4)
    "prior_mean_gap_days": lambda values: values < 0,                   # dias negativos (-4)
    "prior_mean_purchase_value": lambda values: (values < 0) | (values == 1e9),
    "spend_365d": lambda values: (values < 0) | (values == 1e9),
}


def invalid_flags(frame):
    """Uma coluna por regra, True onde o valor é impossível (como o domain_flags da W3).

    Mais a regra de duas colunas da EDA: uma média de compras não pode ser maior
    do que a maior dessas compras.
    """
    flags = pd.concat([rule(frame[c]) for c, rule in invalid_errors.items()], axis=1)
    flags["prior_mean_purchase_value"] |= (frame["prior_mean_purchase_value"]
                                           > frame["prior_max_purchase_value"] + 0.02)
    return flags


def blank_invalid(frame):
    """Os valores impossíveis passam a NaN (W3: "blank the cell").

    Não aprende nada com os dados, por isso aplica-se igual ao treino, à validação e ao teste.
    """
    out = frame.copy()
    out[list(invalid_errors)] = out[list(invalid_errors)].mask(invalid_flags(frame))
    return out


flags = invalid_flags(frame)
print("valores impossíveis por coluna:", flags.sum().to_dict())
print("linhas com algum valor impossível:", int(flags.any(axis=1).sum()), "de", len(frame))

valores impossíveis por coluna: {'days_since_previous_purchase': 16, 'prior_mean_gap_days': 16, 'prior_mean_purchase_value': 77, 'spend_365d': 30}
linhas com algum valor impossível: 139 de 6469


In [10]:
# Passo 8 - manter, passar a nulo ou apagar a linha? (W3, células 63 a 69)
# Nos três casos os nulos são depois preenchidos como na linha de base (mediana / moda).
def prepare_blank(train, test):
    """Valores impossíveis passam a nulo, nas duas partes."""
    return prepare_base(blank_invalid(train), blank_invalid(test))


def prepare_drop(train, test):
    """Apaga do TREINO as linhas com valores impossíveis (W3: drop_invalid_rows).

    A validação nunca perde linhas, porque todas as compras do teste precisam de previsão.
    """
    keep = ~invalid_flags(train).any(axis=1)
    return prepare_base(train[keep], blank_invalid(test))


invalid_runs = {"passar a nulo": evaluate(prepare_blank, frame, splits),
                "apagar a linha (treino)": evaluate(prepare_drop, frame, splits)}
compare(invalid_runs, baseline).round(4)

,f1,accuracy,vs base,SEM,melhor em
passar a nulo,0.734164,0.707607,0.00206,0.0011,4 de 5
apagar a linha (treino),0.734751,0.707421,0.002646,0.001175,5 de 5


In [11]:
# ===== [2.1] VALORES IMPOSSÍVEIS - fim =====
# Passo 9 - o que fica decidido no 2.1:
#   - valores impossíveis -> passar a nulo (blank_invalid); os nulos são tratados no 2.2
#   - as 65 linhas vazias (só cliente e alvo) ficam fora do treino; já estão fora das
#     divisões desde o passo 1, e o test.csv não tem linhas assim
print("linhas do teste sem data:", int(test_csv[DATE].isna().sum()))
print("valores impossíveis no teste (a regra aplica-se igual):",
      int(invalid_flags(test_csv).sum().sum()))

linhas do teste sem data: 0
valores impossíveis no teste (a regra aplica-se igual): 18


In [12]:
# ===== [2.2] VALORES EM FALTA - início =====
# Passo 10 - nulos que sobram depois do 2.1
after_21 = blank_invalid(frame)
nulls = after_21.isna().sum()
nulls[nulls > 0].sort_values(ascending=False).to_frame("nulos")

,nulos
prior_std_gap_days,1413
prior_mean_gap_days,1259
prior_std_purchase_value,1048
prior_mean_purchase_value,874
days_since_previous_purchase,812
prior_min_purchase_value,603
prior_max_purchase_value,603
spend_365d,225
transport_method,196
payment_type,195


In [13]:
# Passo 11 - os nulos estruturais (EDA, passo 12): o valor ainda não pode existir porque
# o cliente tem poucas compras anteriores. Mínimo de compras anteriores para cada coluna existir:
needs_history = {
    "days_since_previous_purchase": 1,
    "prior_mean_purchase_value": 1,
    "prior_min_purchase_value": 1,
    "prior_max_purchase_value": 1,
    "prior_std_purchase_value": 2,   # não há desvio-padrão de um só valor
    "prior_mean_gap_days": 2,        # um intervalo precisa de duas compras
    "prior_std_gap_days": 3,         # o desvio dos intervalos precisa de dois intervalos
}


def zero_where_none(frame):
    """Zero onde o valor ainda não existe; os outros nulos ficam para a mediana.

    É o plano "zero where none / median elsewhere" da W3 (célula 79). Não aprende
    nada com os dados, por isso aplica-se igual ao treino e ao teste.
    """
    out = frame.copy()
    for column, minimum in needs_history.items():
        no_history = out[column].isna() & (out["prior_purchase_count"] < minimum)
        out.loc[no_history, column] = 0
    return out


def own_level(frame):
    """Nulos das categóricas passam a um nível próprio, "(missing)" (W3, célula 79)."""
    out = frame.copy()
    _, categorical = model_columns(out)
    out[categorical] = out[categorical].fillna("(missing)")
    return out


structural = int(after_21.isna().sum().sum() - zero_where_none(after_21).isna().sum().sum())
print(f"nulos no total: {int(after_21.isna().sum().sum()):,}, dos quais estruturais: {structural:,}")

nulos no total: 7,617, dos quais estruturais: 5,921


In [14]:
# Passo 12 - comparar planos de preenchimento (W3, células 79 a 87), todos depois do 2.1
def prepare_zero_where_none(train, test):
    return prepare_base(zero_where_none(blank_invalid(train)), zero_where_none(blank_invalid(test)))


def prepare_own_level(train, test):
    return prepare_base(own_level(blank_invalid(train)), own_level(blank_invalid(test)))


fill_runs = {"mediana / moda": invalid_runs["passar a nulo"],
             "zero sem histórico, mediana no resto": evaluate(prepare_zero_where_none, frame, splits),
             "mediana / nível próprio": evaluate(prepare_own_level, frame, splits)}
compare(fill_runs, invalid_runs["passar a nulo"]).round(4)

,f1,accuracy,vs base,SEM,melhor em
mediana / moda,0.734164,0.707607,0.0,0.0,0 de 5
"zero sem histórico, mediana no resto",0.734237,0.707421,0.000074,0.000404,2 de 5
mediana / nível próprio,0.734468,0.707607,0.000304,0.001256,2 de 5


In [15]:
# ===== [2.2] VALORES EM FALTA - fim =====
# Passo 13 - o que fica decidido no 2.1 e no 2.2, numa só preparação para os pacotes seguintes:
#   1. valores impossíveis -> nulo (blank_invalid), igual no treino e no teste
#   2. nulos -> mediana nas numéricas e moda nas categóricas, ajustadas só no treino
#      (as alternativas ficaram dentro do ruído; como na W3, fica a mais simples)
prepare_clean = prepare_blank

clean = invalid_runs["passar a nulo"]
print(f"2.1 + 2.2: F1 {clean['f1'].mean():.4f} +- {sem(clean['f1']):.4f}"
      f" | accuracy {clean['accuracy'].mean():.4f} +- {sem(clean['accuracy']):.4f}")

2.1 + 2.2: F1 0.7342 +- 0.0093 | accuracy 0.7076 +- 0.0024
